# kv-bench on Kaggle — *Bytes, Not Ratios*

Self-contained. **Before running:** open the right-hand panel →
**Settings → Accelerator = GPU T4 x2** (or P100) and **Internet = On**.
Then **Run All**. Results + figures are written to `/kaggle/working` (saved as output when
you *Save Version*) and zipped into `kvbench_artifacts.zip` for download.

Runtime: roughly 1–3 h for the full 3B+7B sweep. To shorten, edit `configs/kaggle.yaml`
(fewer models / context lengths / `n_items`) in the config cell below before running.


## 1. Install

In [ ]:
# kvpress pulls a compatible torch/transformers. Quiet install.
!pip install -q kvpress matplotlib pyyaml
import torch
print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Enable GPU: Settings -> Accelerator -> GPU'


## 2. Write the kvbench sources (exact repo layout)

In [ ]:
import os
for d in ['src/kvbench', 'scripts', 'configs', 'results', 'reports/figures']:
    os.makedirs(f'/kaggle/working/{d}', exist_ok=True)


In [ ]:
%%writefile /kaggle/working/src/kvbench/__init__.py
"""kvbench — byte-accounting audit of KV cache compression methods.

The thesis: the field budgets compression in tokens and reports it as memory savings; those are
not the same quantity. This package measures realized bytes directly (bytes.py), classifies each
method into a memory-faithfulness taxonomy (presses.py), and drives a resumable sweep (run.py).

Built on NVIDIA kvpress (vendored) + HuggingFace transformers >=5. Device-agnostic: MPS today,
CUDA on the Ubuntu rerun.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile /kaggle/working/src/kvbench/device.py
"""Device selection and synchronization — device-agnostic across MPS / CUDA / CPU.

SRS FR4: every timed region must contain an explicit device sync. MPS and CUDA dispatch
asynchronously; without a sync you measure queue-submission time, not execution. This module is
the single place that knows which backend is live, so timing and memory code stay portable between
the local Mac (MPS) and the later Ubuntu box (CUDA).
"""
from __future__ import annotations

import torch


def pick_device(prefer: str | None = None) -> torch.device:
    """Return the best available device, or the requested one if available."""
    if prefer:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def synchronize(device: torch.device) -> None:
    """Block until all queued work on `device` has executed. No-op on CPU."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def reset_peak_memory(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    # MPS has no peak-reset; we sample current_allocated_memory around regions instead.


def peak_allocated_bytes(device: torch.device) -> int | None:
    """Peak allocator bytes since last reset, or None if the backend can't report it."""
    if device.type == "cuda":
        return int(torch.cuda.max_memory_allocated())
    if device.type == "mps":
        # MPS exposes current, not peak; callers sample this at the region's high-water point.
        return int(torch.mps.current_allocated_memory())
    return None


def current_allocated_bytes(device: torch.device) -> int | None:
    if device.type == "cuda":
        return int(torch.cuda.memory_allocated())
    if device.type == "mps":
        return int(torch.mps.current_allocated_memory())
    return None


def empty_cache(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def input_device(model, fallback: torch.device) -> torch.device:
    """Device to place input_ids on. With device_map sharding, use the embedding's device;
    otherwise the single device the model lives on."""
    try:
        emb = model.get_input_embeddings()
        if emb is not None and hasattr(emb, "weight"):
            return emb.weight.device
    except Exception:  # noqa: BLE001
        pass
    return fallback


def load_model(model_name, dtype, attn_impl, device, device_map=None):
    """Load a causal LM portably across single-device (.to) and multi-GPU (device_map=auto).

    Uses `dtype=` (not the deprecated `torch_dtype=`). When device_map is set (e.g. "auto" on a
    Kaggle T4x2), transformers shards the model across GPUs and we do NOT call .to(device).
    """
    from transformers import AutoModelForCausalLM

    kwargs = dict(dtype=dtype, attn_implementation=attn_impl)
    if device_map:
        model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device_map, **kwargs)
        return model.eval()
    return AutoModelForCausalLM.from_pretrained(model_name, **kwargs).to(device).eval()


In [ ]:
%%writefile /kaggle/working/src/kvbench/bytes.py
"""The measurement instrument: realized bytes of a KV cache, read from stored tensors.

This is the paper's contribution in code. Every other benchmark infers memory from a nominal token
count; we measure the tensors that actually exist. A method that masks without slicing has
full-size key/value tensors and therefore full payload, regardless of the compression ratio it
reports — which is exactly how this instrument exposes simulation-only compression.

Realized bytes (paper Eq. 1):

    bytes(m) = sum_layers (|K_l| + |V_l|) * itemsize     # payload, from the STORED tensor
             + meta(m)                                    # accumulators, masks, tables

Sanity control: for a full cache this must reproduce the analytic
    2 * n_layers * n_kv_heads * head_dim * n_tokens * itemsize.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import torch


@dataclass
class ByteReport:
    payload_bytes: int
    meta_bytes: int
    n_layers: int
    n_kv_heads: int
    head_dim: int
    seq_len: int
    itemsize: int
    per_layer_tokens: list[int] = field(default_factory=list)
    meta_breakdown: dict[str, int] = field(default_factory=dict)

    @property
    def total_bytes(self) -> int:
        return self.payload_bytes + self.meta_bytes

    def as_dict(self) -> dict:
        return {
            "payload_bytes": self.payload_bytes,
            "meta_bytes": self.meta_bytes,
            "total_bytes": self.total_bytes,
            "n_layers": self.n_layers,
            "n_kv_heads": self.n_kv_heads,
            "head_dim": self.head_dim,
            "seq_len": self.seq_len,
            "itemsize": self.itemsize,
            "per_layer_tokens": self.per_layer_tokens,
            "meta_breakdown": self.meta_breakdown,
        }


def _iter_layers(cache):
    """Yield (keys, values) per layer across the transformers>=5 DynamicCache API and older ones.

    transformers>=5: cache.layers[i].keys / .values
    older:           cache.key_cache[i] / cache.value_cache[i]
    """
    layers = getattr(cache, "layers", None)
    if layers is not None:
        for layer in layers:
            k = getattr(layer, "keys", None)
            v = getattr(layer, "values", None)
            if k is None or v is None:
                continue
            yield k, v
        return
    key_cache = getattr(cache, "key_cache", None)
    value_cache = getattr(cache, "value_cache", None)
    if key_cache is not None and value_cache is not None:
        for k, v in zip(key_cache, value_cache):
            if k is None or v is None:
                continue
            yield k, v
        return
    raise TypeError(f"Unrecognized cache type {type(cache).__name__}: no .layers or .key_cache")


def payload_bytes(cache) -> tuple[int, list[int], dict]:
    """Bytes actually occupied by K/V tensors, plus per-layer token counts and shape info."""
    total = 0
    per_layer_tokens: list[int] = []
    shape_info: dict = {}
    for k, v in _iter_layers(cache):
        # [batch, n_kv_heads, seq, head_dim]
        total += k.numel() * k.element_size()
        total += v.numel() * v.element_size()
        per_layer_tokens.append(int(k.shape[-2]))
        if not shape_info:
            shape_info = {
                "n_kv_heads": int(k.shape[1]),
                "head_dim": int(k.shape[-1]),
                "itemsize": int(k.element_size()),
                "batch": int(k.shape[0]),
            }
    return total, per_layer_tokens, shape_info


def meta_bytes(cache, module_list=None) -> tuple[int, dict]:
    """Bytes of per-method bookkeeping that a token-fraction budget ignores.

    Covers the state kvpress attaches to attention modules:
      - masked_key_indices  (simulation-only presses: AdaKV, kvzip, criticalkv, duo, ...)
      - any accumulator/score tensors a press stashes on the module

    A simulation-only press keeps full payload AND carries mask index tensors — both are real
    resident bytes that the reported compression ratio omits. We surface them rather than hide them.
    """
    breakdown: dict[str, int] = {}
    total = 0
    if module_list is None:
        return 0, breakdown
    for mod in module_list:
        mki = getattr(mod, "masked_key_indices", None)
        if mki is not None:
            b = 0
            seq = mki if isinstance(mki, (tuple, list)) else [mki]
            for t in seq:
                if isinstance(t, torch.Tensor):
                    b += t.numel() * t.element_size()
            breakdown["masked_key_indices"] = breakdown.get("masked_key_indices", 0) + b
            total += b
    return total, breakdown


def measure(cache, module_list=None) -> ByteReport:
    """Full realized-byte report for a cache (paper Eq. 1)."""
    payload, per_layer_tokens, shape = payload_bytes(cache)
    meta, meta_break = meta_bytes(cache, module_list)
    return ByteReport(
        payload_bytes=payload,
        meta_bytes=meta,
        n_layers=len(per_layer_tokens),
        n_kv_heads=shape.get("n_kv_heads", 0),
        head_dim=shape.get("head_dim", 0),
        seq_len=max(per_layer_tokens) if per_layer_tokens else 0,
        itemsize=shape.get("itemsize", 0),
        per_layer_tokens=per_layer_tokens,
        meta_breakdown=meta_break,
    )


def analytic_full_cache_bytes(
    n_layers: int, n_kv_heads: int, head_dim: int, n_tokens: int, itemsize: int, batch: int = 1
) -> int:
    """The full-cache byte figure a correct instrument must reproduce (sanity control)."""
    return 2 * n_layers * n_kv_heads * head_dim * n_tokens * itemsize * batch


def collect_attention_modules(model) -> list:
    """Attention submodules kvpress may attach state to (for meta-byte accounting)."""
    mods = []
    for _, module in model.named_modules():
        # kvpress attaches masked_key_indices to self_attn modules; match by attribute surface.
        if hasattr(module, "masked_key_indices") or module.__class__.__name__.endswith("Attention"):
            mods.append(module)
    return mods


In [ ]:
%%writefile /kaggle/working/src/kvbench/presses.py
"""Press registry, memory-faithfulness taxonomy, and the byte-matched budget wrapper.

The taxonomy is the paper's organizing contribution. We assign a *prior* class by reading the
kvpress source (which compression path each press takes), and the audit runner *confirms* it
dynamically by measuring realized bytes. Prior and measurement must agree; a disagreement is a
finding.
"""
from __future__ import annotations

from dataclasses import dataclass

from kvpress import (
    AdaKVPress,
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    ThinKPress,
    TOVAPress,
)

# Taxonomy prior, from source inspection of kvpress 0.5.4:
#   memory-faithful  : ScorerPress.compress gathers K/V to n_kept and returns the smaller tensors.
#   simulation-only  : compress sets module.masked_key_indices and returns K/V UNMODIFIED; the
#                      attention_patch masks at compute time. Payload never shrinks.
#   incommensurable  : compresses off the token axis (ThinK prunes head_dim channels), so a
#                      token-fraction ratio is not a statement about token count at all.
MEMORY_FAITHFUL = "memory-faithful"
SIMULATION_ONLY = "simulation-only"
INCOMMENSURABLE = "incommensurable"


@dataclass
class PressSpec:
    name: str
    taxonomy_prior: str
    query_aware: bool  # RQ2 grouping — does scoring peek at an end-of-prompt observation window?
    build: callable    # (compression_ratio) -> press


def _snapkv(r):
    return SnapKVPress(compression_ratio=r)


def _adakv(r):
    # AdaKV wraps a ScorerPress; SnapKV is the canonical inner scorer. AdaKV itself only masks.
    return AdaKVPress(press=SnapKVPress(compression_ratio=r))


REGISTRY: dict[str, PressSpec] = {
    "random": PressSpec("random", MEMORY_FAITHFUL, False, lambda r: RandomPress(compression_ratio=r)),
    "knorm": PressSpec("knorm", MEMORY_FAITHFUL, False, lambda r: KnormPress(compression_ratio=r)),
    "streaming_llm": PressSpec(
        "streaming_llm", MEMORY_FAITHFUL, False, lambda r: StreamingLLMPress(compression_ratio=r)
    ),
    "tova": PressSpec("tova", MEMORY_FAITHFUL, False, lambda r: TOVAPress(compression_ratio=r)),
    "snapkv": PressSpec("snapkv", MEMORY_FAITHFUL, True, _snapkv),
    "expected_attention": PressSpec(
        "expected_attention", MEMORY_FAITHFUL, False,
        lambda r: ExpectedAttentionPress(compression_ratio=r),
    ),
    "observed_attention": PressSpec(
        "observed_attention", MEMORY_FAITHFUL, False,
        lambda r: ObservedAttentionPress(compression_ratio=r),
    ),  # closest thing kvpress ships to H2O (accumulated observed attention)
    "adakv": PressSpec("adakv", SIMULATION_ONLY, True, _adakv),
    "think": PressSpec("think", INCOMMENSURABLE, False, lambda r: ThinKPress(key_channel_compression_ratio=r)),
}


def build_press(name: str, ratio: float):
    if name not in REGISTRY:
        raise KeyError(f"unknown press '{name}'; known: {sorted(REGISTRY)}")
    return REGISTRY[name].build(ratio)


def classify_measured(full_cache_bytes: int, realized_bytes: int, tol: float = 0.02) -> str:
    """Dynamic taxonomy from measurement: near-full payload at high ratio => simulation-only."""
    frac = realized_bytes / full_cache_bytes if full_cache_bytes else 1.0
    if frac >= 1.0 - tol:
        return SIMULATION_ONLY
    return MEMORY_FAITHFUL  # incommensurable is decided by construction (ThinK), not by this test


def calibrate_ratio_for_byte_fraction(
    measure_fn, target_fraction: float, tol: float = 0.01, max_iter: int = 12
) -> tuple[float | None, float]:
    """Find the nominal compression_ratio whose realized byte fraction hits `target_fraction`.

    `measure_fn(ratio) -> realized_fraction` runs a prefill and measures realized/full bytes.
    Returns (ratio, achieved_fraction). ratio is None if the target is INFEASIBLE — i.e. even the
    most aggressive ratio cannot get realized bytes down to the target (the signature of a
    simulation-only method, which cannot be byte-matched below full cache). We report that as a
    finding rather than silently substituting the closest ratio (SRS NFR4: honest failure).

    Assumes realized_fraction is monotone non-increasing in ratio (true for memory-faithful presses;
    for masking presses the fraction stays ~1 and the infeasibility branch fires).
    """
    lo, hi = 0.0, 0.99
    f_hi = measure_fn(hi)  # most aggressive
    if f_hi > target_fraction + tol:
        return None, f_hi  # cannot reach target even at max compression -> infeasible
    f_lo = measure_fn(lo)
    if f_lo <= target_fraction:  # full cache already at/under target (target ~1.0)
        return lo, f_lo
    best = hi
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        f = measure_fn(mid)
        if abs(f - target_fraction) <= tol:
            return mid, f
        if f > target_fraction:  # not compressed enough
            lo = mid
        else:
            hi, best = mid, mid
    return best, measure_fn(best)


In [ ]:
%%writefile /kaggle/working/src/kvbench/tasks.py
"""Self-contained synthetic long-context task — no dataset download.

A RULER-style needle-in-a-haystack (key/value retrieval): a haystack of filler sentences with one
planted "magic number" fact, and a question at the END asking to recall it. Question-at-end matches
the benchmark convention this project critiques and the kvpress default, so it is the fair setting
for query-aware methods (SnapKV) — we are not stacking the deck against them.

Deterministic given a seed, so runs are reproducible (SRS NFR1). The real extended sweep swaps this
for RULER / LongBench via kvpress's own eval loop; this keeps the local pilot dependency-free.
"""
from __future__ import annotations

import random
from dataclasses import dataclass

FILLER = (
    "The garden was quiet in the afternoon light. "
    "A gentle breeze moved through the tall grass. "
    "Somewhere nearby a bird called out twice. "
    "The old clock on the wall ticked steadily. "
    "Rain had fallen earlier and the ground was damp. "
)


@dataclass
class Needle:
    context: str
    question: str
    answer: str  # the exact string the model must reproduce


def make_needle(
    tokenizer,
    context_tokens: int,
    seed: int = 0,
    depth: float = 0.5,
) -> Needle:
    """Build a needle-in-a-haystack item ~context_tokens long, needle at fractional `depth`."""
    rng = random.Random(seed)
    magic = rng.randint(1000000, 9999999)
    label = rng.choice(["alpha", "bravo", "delta", "orion", "vega", "lyra"])
    needle_sentence = f" The secret code for {label} is {magic}. "

    # Build filler to roughly context_tokens, insert needle at `depth`.
    approx_sentences = max(4, context_tokens // 12)
    body = (FILLER * (approx_sentences // 5 + 1)).split(". ")
    insert_at = int(len(body) * depth)
    body.insert(insert_at, needle_sentence.strip())
    context = ". ".join(s for s in body if s).strip()

    # Trim to token budget.
    ids = tokenizer(context, return_tensors="pt").input_ids[0]
    if ids.shape[0] > context_tokens:
        ids = ids[:context_tokens]
        context = tokenizer.decode(ids, skip_special_tokens=True)
        if str(magic) not in context:
            # needle got trimmed; re-insert near the (now shorter) middle
            context = context[: len(context) // 2] + needle_sentence + context[len(context) // 2 :]

    question = f"What is the secret code for {label}? Answer with the number only."
    return Needle(context=context, question=question, answer=str(magic))


def score(prediction: str, answer: str) -> float:
    """1.0 if the exact answer string appears in the prediction, else 0.0."""
    return 1.0 if answer in prediction else 0.0


In [ ]:
%%writefile /kaggle/working/src/kvbench/run.py
"""Phase-2 audit runner: measure realized KV bytes per (model, context_len, press, ratio).

No accuracy evaluation here — this is the cheap, decisive figure. For each cell we prefill a fixed
context, apply the press, and measure the realized bytes of the resulting cache against the
full-cache baseline. Writes one JSON per cell to results/; resumable (skips existing cells).

  python -m kvbench.run --config configs/pilot.yaml

Device-agnostic (MPS/CUDA/CPU) via kvbench.device.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import time

import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer

from kvbench import bytes as kb
from kvbench import device as kvdev
from kvbench.presses import REGISTRY, build_press, classify_measured

# Attention-score methods need the materialized attention matrix, so they cannot use SDPA/Flash
# kernels — they require eager attention. That incompatibility is itself a systems finding
# (cf. KeyDiff, arXiv:2504.15364); we record it rather than hide it.
EAGER_METHODS = {"observed_attention"}


def cell_id(model: str, ctx: int, press: str, ratio: float) -> str:
    safe_model = model.replace("/", "_")
    return f"{safe_model}__ctx{ctx}__{press}__r{ratio:.3f}"


def full_cache_baseline(model, tokenizer, input_ids, device) -> kb.ByteReport:
    with torch.no_grad():
        out = model(input_ids, use_cache=True)
    return kb.measure(out.past_key_values)


def run_cell(model, tokenizer, modules, input_ids, ctx, press_name, ratio, device) -> dict:
    spec = REGISTRY[press_name]
    press = build_press(press_name, ratio)
    if hasattr(press, "post_init_from_model"):
        try:
            press.post_init_from_model(model)
        except Exception:  # noqa: BLE001 — some presses init lazily via the hook
            pass

    kvdev.empty_cache(device)
    kvdev.synchronize(device)
    with torch.no_grad(), press(model):
        out = model(input_ids, use_cache=True)
    kvdev.synchronize(device)

    rep = kb.measure(out.past_key_values, module_list=modules)
    full = full_cache_baseline(model, tokenizer, input_ids, device)
    measured_class = classify_measured(full.total_bytes, rep.total_bytes)

    return {
        "method": press_name,
        "model": model.name_or_path,
        "context_len": int(input_ids.shape[-1]),
        "requested_context_len": ctx,
        "nominal_ratio": ratio,
        "realized_bytes": rep.total_bytes,
        "payload_bytes": rep.payload_bytes,
        "meta_bytes": rep.meta_bytes,
        "meta_breakdown": rep.meta_breakdown,
        "full_cache_bytes": full.total_bytes,
        "realized_fraction": rep.total_bytes / full.total_bytes if full.total_bytes else None,
        "taxonomy_prior": spec.taxonomy_prior,
        "taxonomy_measured": measured_class,
        "query_aware": spec.query_aware,
        "per_layer_tokens": rep.per_layer_tokens,
        "n_kv_heads": rep.n_kv_heads,
        "head_dim": rep.head_dim,
        "itemsize": rep.itemsize,
        "device": device.type,
        "torch_version": torch.__version__,
        "timestamp": time.time(),
    }


def make_context(tokenizer, ctx_len: int, device) -> torch.Tensor:
    """Deterministic filler context of exactly ctx_len tokens (audit needs length, not meaning)."""
    text = ("The quick brown fox jumps over the lazy dog. " * ((ctx_len // 8) + 2))
    ids = tokenizer(text, return_tensors="pt").input_ids[:, :ctx_len]
    return ids.to(device)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--results", default="results")
    ap.add_argument("--device", default=None, help="force mps/cuda/cpu")
    args = ap.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)
    os.makedirs(args.results, exist_ok=True)
    device = kvdev.pick_device(args.device)
    print(f"device: {device}")

    dtype = torch.float16 if device.type in ("cuda", "mps") else torch.float32

    device_map = cfg.get("device_map")
    for model_name in cfg["models"]:
        # Load ONE model per model_name. If any method needs eager attention, use eager for all —
        # holding two sharded models at once under device_map=auto risks a CUDA OOM/launch failure.
        # Attention impl does not change realized byte measurements, so this is safe for the audit.
        attn_impl = "eager" if any(m in EAGER_METHODS for m in cfg["methods"]) else "sdpa"
        print(f"loading {model_name} (attn={attn_impl}, device_map={device_map}) ...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = kvdev.load_model(model_name, dtype, attn_impl, device, device_map)
        in_dev = kvdev.input_device(model, device)

        for ctx in cfg["context_lengths"]:
            for press_name in cfg["methods"]:
                modules = kb.collect_attention_modules(model)
                input_ids = make_context(tokenizer, ctx, in_dev)
                for ratio in cfg["ratios"]:
                    cid = cell_id(model_name, ctx, press_name, ratio)
                    path = os.path.join(args.results, cid + ".json")
                    if os.path.exists(path):
                        print(f"  skip {cid}")
                        continue
                    try:
                        rec = run_cell(model, tokenizer, modules, input_ids, ctx, press_name, ratio, device)
                        with open(path, "w") as f:
                            json.dump(rec, f, indent=2)
                        frac = rec["realized_fraction"]
                        print(f"  {cid}: realized={frac:.3f} of full  [{rec['taxonomy_measured']}]")
                    except Exception as e:  # noqa: BLE001 — record honest failure, never interpolate
                        fail = {"cell": cid, "error": repr(e), "failed": True}
                        with open(path, "w") as f:
                            json.dump(fail, f, indent=2)
                        print(f"  {cid}: FAILED {e!r}")

        del model
        kvdev.empty_cache(device)

    print("done.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/src/kvbench/eval.py
"""Phase-3 accuracy runner: task accuracy at matched BYTE budgets (not matched token fractions).

For each method we calibrate the nominal compression_ratio that hits each target byte fraction,
then evaluate accuracy on the synthetic needle task using the kvpress generation pipeline (which
compresses the context during prefill and excludes the question from compression). Masking methods
that cannot reach a byte target are recorded as INFEASIBLE, never interpolated.

  python -m kvbench.eval --config configs/eval_pilot.yaml

Writes one JSON per (model, ctx, method, byte_fraction) cell to results/. Device-agnostic.
"""
from __future__ import annotations

import argparse
import json
import os
import time

import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer

from kvbench import bytes as kb
from kvbench import device as kvdev
from kvbench.presses import (
    REGISTRY,
    build_press,
    calibrate_ratio_for_byte_fraction,
)
from kvbench.run import EAGER_METHODS
from kvbench.tasks import make_needle, score


def measure_fraction_fn(model, tokenizer, modules, context, device):
    """Return f(ratio)->realized_fraction by prefilling `context` under the press at that ratio."""
    in_dev = kvdev.input_device(model, device)
    ids = tokenizer(context, return_tensors="pt").input_ids.to(in_dev)
    with torch.no_grad():
        full = kb.measure(model(ids, use_cache=True).past_key_values)

    def f(ratio: float) -> float:
        if ratio <= 0:
            return 1.0
        press = build_press(method_name.value, ratio)
        if hasattr(press, "post_init_from_model"):
            try:
                press.post_init_from_model(model)
            except Exception:  # noqa: BLE001
                pass
        with torch.no_grad(), press(model):
            rep = kb.measure(model(ids, use_cache=True).past_key_values, module_list=modules)
        return rep.total_bytes / full.total_bytes if full.total_bytes else 1.0

    return f, full.total_bytes


class _Box:
    """Tiny mutable holder so the closure above can see the current method name."""

    value: str = ""


method_name = _Box()


def _eval_accuracy(pipe, tokenizer, ctx, meth, ratio, n_items) -> float:
    press = build_press(meth, ratio)
    if hasattr(press, "post_init_from_model"):
        try:
            press.post_init_from_model(pipe.model)
        except Exception:  # noqa: BLE001
            pass
    hits, n = 0.0, 0
    for seed in range(n_items):
        depth = (seed + 1) / (n_items + 1)
        item = make_needle(tokenizer, ctx, seed=seed, depth=depth)
        out = pipe(item.context, question=item.question, press=press, max_new_tokens=16)
        pred = out["answers"][0] if isinstance(out.get("answers"), list) else str(out)
        hits += score(pred, item.answer)
        n += 1
    return hits / n if n else 0.0


def run(model, tokenizer, modules, cfg, model_name, device, results_dir):
    """Fixed-nominal-ratio accuracy + realized bytes per cell.

    Records BOTH accuracy and realized byte fraction at each nominal ratio, so the analysis can plot
    accuracy against the field's axis (nominal ratio) AND the honest axis (realized bytes) from the
    same cells — that contrast is Fig 2. Simulation-only methods (AdaKV) appear at realized~1.0
    regardless of ratio, exposing them on the byte axis.
    """
    from kvpress import KVPressTextGenerationPipeline

    pipe = KVPressTextGenerationPipeline(model=model, tokenizer=tokenizer)
    n_items = cfg.get("n_items", 5)

    for ctx in cfg["context_lengths"]:
        calib = make_needle(tokenizer, ctx, seed=999, depth=0.5).context
        for meth in cfg["methods"]:
            method_name.value = meth
            frac_fn, full_bytes = measure_fraction_fn(model, tokenizer, modules, calib, device)
            for ratio in cfg["nominal_ratios"]:
                cid = f"acc__{model_name.replace('/', '_')}__ctx{ctx}__{meth}__r{ratio:.2f}"
                path = os.path.join(results_dir, cid + ".json")
                if os.path.exists(path):
                    print(f"  skip {cid}")
                    continue
                try:
                    realized_frac = frac_fn(ratio)
                    acc = _eval_accuracy(pipe, tokenizer, ctx, meth, ratio, n_items)
                    rec = {
                        "cell": cid, "kind": "accuracy", "method": meth, "model": model_name,
                        "context_len": ctx, "nominal_ratio": ratio,
                        "realized_fraction": realized_frac,
                        "realized_bytes": int(realized_frac * full_bytes),
                        "full_cache_bytes": full_bytes, "accuracy": acc, "n_items": n_items,
                        "query_aware": REGISTRY[meth].query_aware,
                        "taxonomy_prior": REGISTRY[meth].taxonomy_prior,
                        "device": device.type, "timestamp": time.time(),
                    }
                    with open(path, "w") as f:
                        json.dump(rec, f, indent=2)
                    print(f"  {cid}: realized={realized_frac:.2f} acc={acc:.2f}")
                except Exception as e:  # noqa: BLE001 — honest failure, never interpolate
                    with open(path, "w") as f:
                        json.dump({"cell": cid, "error": repr(e), "failed": True}, f, indent=2)
                    print(f"  {cid}: FAILED {e!r}")


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--results", default="results")
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)
    os.makedirs(args.results, exist_ok=True)
    device = kvdev.pick_device(args.device)
    dtype = torch.float16 if device.type in ("cuda", "mps") else torch.float32
    print(f"device: {device}")

    device_map = cfg.get("device_map")  # e.g. "auto" for Kaggle T4x2 / multi-GPU
    for model_name in cfg["models"]:
        needs_eager = any(m in EAGER_METHODS for m in cfg["methods"])
        attn = "eager" if needs_eager else "sdpa"
        print(f"loading {model_name} (attn={attn}, device_map={device_map}) ...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = kvdev.load_model(model_name, dtype, attn, device, device_map)
        modules = kb.collect_attention_modules(model)
        run(model, tokenizer, modules, cfg, model_name, device, args.results)
        del model
        kvdev.empty_cache(device)

    print("done.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/src/kvbench/selftest.py
"""Instrument self-tests — run before trusting any measurement.

  python -m kvbench.selftest

Checks (paper §Method sanity control):
  1. On a synthetic full cache with known dims, measure() reproduces the analytic byte figure exactly.
  2. Physically slicing K/V (memory-faithful path) reduces measured payload proportionally.
  3. Masking without slicing (simulation-only path) leaves payload unchanged — the positive control.
"""
from __future__ import annotations

import sys

import torch
from transformers import DynamicCache

from kvbench import bytes as kb


def _make_cache(n_layers, batch, n_kv_heads, seq, head_dim, dtype=torch.float32) -> DynamicCache:
    c = DynamicCache()
    for layer in range(n_layers):
        k = torch.zeros(batch, n_kv_heads, seq, head_dim, dtype=dtype)
        v = torch.zeros(batch, n_kv_heads, seq, head_dim, dtype=dtype)
        c.update(k, v, layer)
    return c


def main() -> int:
    n_layers, batch, n_kv_heads, seq, head_dim = 4, 1, 8, 128, 64
    itemsize = torch.finfo(torch.float32).bits // 8

    # 1. analytic reproduction
    c = _make_cache(n_layers, batch, n_kv_heads, seq, head_dim)
    rep = kb.measure(c)
    analytic = kb.analytic_full_cache_bytes(n_layers, n_kv_heads, head_dim, seq, itemsize, batch)
    ok1 = rep.payload_bytes == analytic
    print(f"[1] payload={rep.payload_bytes} analytic={analytic}  {'OK' if ok1 else 'MISMATCH'}")

    # 2. memory-faithful: physically gather to half the tokens
    kept = seq // 2
    for layer in c.layers:
        idx = torch.arange(kept).view(1, 1, kept, 1).expand(batch, n_kv_heads, kept, head_dim)
        layer.keys = layer.keys.gather(2, idx).contiguous()
        layer.values = layer.values.gather(2, idx).contiguous()
    rep2 = kb.measure(c)
    ok2 = rep2.payload_bytes == analytic // 2
    print(f"[2] sliced payload={rep2.payload_bytes} expected={analytic // 2}  "
          f"{'OK' if ok2 else 'MISMATCH'}")

    # 3. simulation-only positive control: full cache + a mask index tensor => payload unchanged,
    #    meta > 0. This is exactly AdaKVPress's behaviour.
    c3 = _make_cache(n_layers, batch, n_kv_heads, seq, head_dim)

    class _FakeAttn:
        pass

    mod = _FakeAttn()
    n_pruned = n_kv_heads * (seq - kept)
    mod.masked_key_indices = (
        torch.arange(batch).repeat_interleave(n_pruned),
        torch.zeros(n_pruned, dtype=torch.long),
        torch.zeros(n_pruned, dtype=torch.long),
    )
    rep3 = kb.measure(c3, module_list=[mod])
    ok3 = rep3.payload_bytes == analytic and rep3.meta_bytes > 0
    print(f"[3] masked payload={rep3.payload_bytes} (== full {analytic}), "
          f"meta={rep3.meta_bytes}  {'OK' if ok3 else 'MISMATCH'}")
    print("    ^ positive control: a masking method reports compression but frees ZERO payload bytes.")

    passed = ok1 and ok2 and ok3
    print(f"\n{'ALL PASS' if passed else 'FAILURES PRESENT'}")
    return 0 if passed else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/scripts/_figlib.py
"""Shared helpers for figure scripts.

Enforces the project's non-negotiable: a figure renders from real result files or it does not
render. `load_cells` raises (non-zero exit) if the results directory is missing or empty, so the
Makefile's `make paper` fails loudly rather than emitting a plot built on nothing.
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import sys


def standard_args(description: str) -> argparse.Namespace:
    ap = argparse.ArgumentParser(description=description)
    ap.add_argument("--results", required=True, help="directory of results/*.json cells")
    ap.add_argument("--out", required=True, help="output PDF path")
    return ap.parse_args()


def load_cells(results_dir: str) -> list[dict]:
    """Load every results/*.json cell. Exit non-zero if none exist — never fabricate."""
    if not os.path.isdir(results_dir):
        sys.exit(f"[figlib] results dir '{results_dir}' does not exist — run experiments first.")
    paths = sorted(glob.glob(os.path.join(results_dir, "*.json")))
    if not paths:
        sys.exit(
            f"[figlib] no result cells in '{results_dir}'. "
            "This figure has no data yet; refusing to emit a placeholder plot. "
            "Run the sweep (src/kvbench/run.py) to populate results/."
        )
    cells = []
    for p in paths:
        with open(p, encoding="utf-8") as f:
            cells.append(json.load(f))
    return cells


# Expected schema of a results/*.json cell (documented here so figure scripts and the runner agree).
# Populated by src/kvbench/run.py once the harness exists:
#   {
#     "method": "snapkv",            # press name
#     "model": "Qwen2.5-7B-Instruct",
#     "context_len": 8192,
#     "nominal_ratio": 0.5,          # the field's knob
#     "realized_bytes": 1234567,     # measured via bytes.py Eq.(1)
#     "full_cache_bytes": 2345678,   # measured full-cache baseline for the same cell
#     "peak_prefill_bytes": 3456789,
#     "taxonomy_class": "simulation-only" | "memory-faithful" | "incommensurable",
#     "byte_budget": 1048576,        # target for byte-matched cells (else null)
#     "accuracy": 0.63,              # task metric
#     "decode_tokens_per_s": 42.0,
#     "seed": 0,
#     "byte_decomposition": {"payload":..., "sinks":..., "recency":..., "obs_window":..., "meta":...}
#   }


In [ ]:
%%writefile /kaggle/working/scripts/fig1_mispricing.py
#!/usr/bin/env python3
"""Fig 1 — nominal compression ratio (x) vs realized KV bytes as a fraction of full cache (y).

The paper's motivating figure. Memory-faithful methods track the diagonal; simulation-only methods
(e.g. masked AdaKV) render as flat lines at ~1.0; sink/observation-window methods diverge upward.
Renders only from real results/*.json — see _figlib.load_cells.
"""
from __future__ import annotations

import sys
from collections import defaultdict

sys.path.insert(0, __file__.rsplit("/", 1)[0])
from _figlib import load_cells, standard_args  # noqa: E402


def main() -> None:
    args = standard_args(__doc__)
    cells = load_cells(args.results)

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    cells = [c for c in cells if not c.get("failed")]
    # One model, one context length for a clean figure: use the largest context present.
    ctx = max(c["context_len"] for c in cells if "context_len" in c)
    cells = [c for c in cells if c.get("context_len") == ctx]

    series: dict[str, list[tuple[float, float]]] = defaultdict(list)
    for c in cells:
        full = c.get("full_cache_bytes")
        if not full:
            continue
        series[c["method"]].append((c["nominal_ratio"], c["realized_bytes"] / full))

    if not series:
        sys.exit("[fig1] no cells carried realized_bytes/full_cache_bytes — cannot plot.")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="ratio promises (payload = fraction)")
    for method, pts in sorted(series.items()):
        pts.sort()
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        ax.plot(xs, ys, marker="o", label=method)
    ax.set_xlabel("nominal compression ratio (fraction of tokens retained)")
    ax.set_ylabel("realized KV bytes / full-cache bytes")
    ax.axhline(1.0, color="0.6", lw=0.8, ls=":")
    ymax = max(1.12, max(y for pts in series.values() for _, y in pts) + 0.03)
    ax.set_ylim(0, ymax)
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
    ax.set_title(f"Ratios are not bytes (ctx={ctx})")
    fig.tight_layout()
    fig.savefig(args.out)
    print(f"[fig1] wrote {args.out} from {len(cells)} cells, {len(series)} methods")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/fig2_reranking.py
#!/usr/bin/env python3
"""Fig 2 — the correction. Two panels from the same accuracy cells:
   LEFT : accuracy vs nominal compression ratio (the field's view).
   RIGHT: accuracy vs realized KV bytes / full cache (the honest view).

Simulation-only methods (AdaKV) look like normal compressors on the left but collapse onto x~1.0
on the right, exposing that they never freed memory. Renders only from real results/acc__*.json.
"""
from __future__ import annotations

import sys
from collections import defaultdict

sys.path.insert(0, __file__.rsplit("/", 1)[0])
from _figlib import load_cells, standard_args  # noqa: E402


def main() -> None:
    args = standard_args(__doc__)
    cells = [c for c in load_cells(args.results) if c.get("kind") == "accuracy" and not c.get("failed")]
    if not cells:
        sys.exit("[fig2] no accuracy cells (results/acc__*.json) — run kvbench.eval first.")

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    ctx = max(c["context_len"] for c in cells)
    cells = [c for c in cells if c["context_len"] == ctx]

    by_nominal: dict[str, list[tuple[float, float]]] = defaultdict(list)
    by_bytes: dict[str, list[tuple[float, float]]] = defaultdict(list)
    for c in cells:
        by_nominal[c["method"]].append((c["nominal_ratio"], c["accuracy"]))
        if c.get("realized_fraction") is not None:
            by_bytes[c["method"]].append((c["realized_fraction"], c["accuracy"]))

    fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
    for method, pts in sorted(by_nominal.items()):
        pts.sort()
        axl.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=method)
    axl.set_xlabel("nominal compression ratio")
    axl.set_ylabel("task accuracy")
    axl.set_title("The field's view (matched nominal ratio)")
    axl.legend(fontsize=8)

    for method, pts in sorted(by_bytes.items()):
        pts.sort()
        axr.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=method)
    axr.axvline(1.0, color="0.6", lw=0.8, ls=":")
    axr.set_xlabel("realized KV bytes / full cache")
    axr.set_title("Corrected (matched realized bytes)")

    fig.suptitle(f"Ratios vs bytes (ctx={ctx})")
    fig.tight_layout()
    fig.savefig(args.out)
    print(f"[fig2] wrote {args.out} from {len(cells)} accuracy cells")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/fig3_decomposition.py
#!/usr/bin/env python3
"""Fig 3 — per-method byte decomposition at a fixed nominal ratio (stacked bar).

Shows WHERE each method's bytes live: payload, sinks, recency window, observation window, metadata,
and the peak-prefill transient. Renders only from real results/*.json.
"""
from __future__ import annotations

import sys
from collections import defaultdict

sys.path.insert(0, __file__.rsplit("/", 1)[0])
from _figlib import load_cells, standard_args  # noqa: E402

COMPONENTS = ["payload", "sinks", "recency", "obs_window", "meta"]


def main() -> None:
    args = standard_args(__doc__)
    cells = load_cells(args.results)

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np

    # One bar per method; take the cell nearest ratio=0.5 for each.
    best: dict[str, dict] = {}
    for c in cells:
        if "byte_decomposition" not in c:
            continue
        m = c["method"]
        if m not in best or abs(c["nominal_ratio"] - 0.5) < abs(best[m]["nominal_ratio"] - 0.5):
            best[m] = c
    if not best:
        sys.exit("[fig3] no cells carried byte_decomposition — cannot plot.")

    methods = sorted(best)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    bottoms = np.zeros(len(methods))
    for comp in COMPONENTS:
        vals = np.array([best[m]["byte_decomposition"].get(comp, 0) for m in methods], float)
        ax.bar(methods, vals, bottom=bottoms, label=comp)
        bottoms += vals
    ax.set_ylabel("bytes")
    ax.set_title("Byte decomposition at nominal ratio ~0.5")
    ax.legend(fontsize=8)
    plt.xticks(rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(args.out)
    print(f"[fig3] wrote {args.out} for {len(methods)} methods")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/fig5_latency.py
#!/usr/bin/env python3
"""Fig 5 — accuracy vs decode throughput at matched bytes (scatter, one point per method/budget).

Tests the systems half of the thesis: methods that free no bytes also buy no speedup, while being
ranked as if they saved both. Renders only from real results/*.json.
"""
from __future__ import annotations

import sys

sys.path.insert(0, __file__.rsplit("/", 1)[0])
from _figlib import load_cells, standard_args  # noqa: E402


def main() -> None:
    args = standard_args(__doc__)
    cells = load_cells(args.results)

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    pts = [
        c
        for c in cells
        if c.get("accuracy") is not None and c.get("decode_tokens_per_s") is not None
    ]
    if not pts:
        sys.exit("[fig5] no cells carried both accuracy and decode_tokens_per_s — run 2b first.")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    methods = sorted({c["method"] for c in pts})
    cmap = plt.get_cmap("tab10")
    for i, m in enumerate(methods):
        mp = [c for c in pts if c["method"] == m]
        ax.scatter(
            [c["decode_tokens_per_s"] for c in mp],
            [c["accuracy"] for c in mp],
            label=m,
            color=cmap(i % 10),
        )
    ax.set_xlabel("decode throughput (tokens/s, device-synced)")
    ax.set_ylabel("task accuracy")
    ax.set_title("Accuracy vs speed at matched bytes")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(args.out)
    print(f"[fig5] wrote {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/scripts/table1_rank.py
#!/usr/bin/env python3
r"""Table 1 — Kendall tau between the token-matched and byte-matched method rankings.

At each budget level we rank methods two ways and compare:
  - matched NOMINAL ratio: rank by accuracy at that nominal ratio.
  - matched REALIZED bytes: rank by accuracy interpolated at the equivalent byte fraction.
tau < 1 means correcting the resource unit reorders the methods — the quantitative form of the
claim. Emits a LaTeX booktabs table to --out (a .tex fragment \input by the paper).

Only memory-faithful methods can be byte-matched below full cache; simulation-only methods that
never reach the byte target are reported separately as INFEASIBLE, not ranked.
"""
from __future__ import annotations

import sys

sys.path.insert(0, __file__.rsplit("/", 1)[0])
from _figlib import load_cells, standard_args  # noqa: E402


def kendall_tau(rank_a: list[str], rank_b: list[str]) -> float | None:
    """Kendall tau-a over the methods common to both orderings. None if <2 common items."""
    common = [m for m in rank_a if m in rank_b]
    if len(common) < 2:
        return None
    pos_b = {m: i for i, m in enumerate(rank_b)}
    order_a = [m for m in rank_a if m in common]
    conc = disc = 0
    for i in range(len(order_a)):
        for j in range(i + 1, len(order_a)):
            s = pos_b[order_a[i]] - pos_b[order_a[j]]
            if s < 0:
                conc += 1
            elif s > 0:
                disc += 1
    n = len(order_a)
    denom = n * (n - 1) / 2
    return (conc - disc) / denom if denom else None


def interp(pts: list[tuple[float, float]], x: float) -> float | None:
    """Linear interpolation of accuracy at byte-fraction x; None if x outside the measured range."""
    pts = sorted(pts)
    if not pts or x < pts[0][0] or x > pts[-1][0]:
        return None
    for (x0, y0), (x1, y1) in zip(pts, pts[1:]):
        if x0 <= x <= x1:
            if x1 == x0:
                return y0
            return y0 + (y1 - y0) * (x - x0) / (x1 - x0)
    return None


def main() -> None:
    args = standard_args(__doc__)
    cells = [c for c in load_cells(args.results) if c.get("kind") == "accuracy" and not c.get("failed")]
    if not cells:
        sys.exit("[table1] no accuracy cells — run kvbench.eval first.")

    ctx = max(c["context_len"] for c in cells)
    cells = [c for c in cells if c["context_len"] == ctx]

    methods = sorted({c["method"] for c in cells})
    nominal_ratios = sorted({c["nominal_ratio"] for c in cells})

    # Per method: nominal->acc and byte_fraction->acc
    nom = {m: [] for m in methods}
    byt = {m: [] for m in methods}
    for c in cells:
        nom[c["method"]].append((c["nominal_ratio"], c["accuracy"]))
        if c.get("realized_fraction") is not None:
            byt[c["method"]].append((c["realized_fraction"], c["accuracy"]))

    rows = []
    for r in nominal_ratios:
        target_byte_fraction = 1.0 - r  # the byte level the field THINKS ratio r delivers
        # nominal ranking: methods by accuracy at nominal ratio r (desc)
        nom_acc = {m: dict(nom[m]).get(r) for m in methods}
        nom_rank = sorted([m for m in methods if nom_acc[m] is not None],
                          key=lambda m: -nom_acc[m])
        # byte ranking: methods by accuracy interpolated at target_byte_fraction (desc)
        byte_acc = {m: interp(byt[m], target_byte_fraction) for m in methods}
        byte_rank = sorted([m for m in methods if byte_acc[m] is not None],
                           key=lambda m: -byte_acc[m])
        infeasible = [m for m in methods if byte_acc[m] is None]
        tau = kendall_tau(nom_rank, byte_rank)
        rows.append((r, target_byte_fraction, tau, infeasible))

    # Emit LaTeX fragment.
    lines = [
        "% Auto-generated by scripts/table1_rank.py — do not hand-edit.",
        "\\begin{tabular}{cccl}",
        "\\toprule",
        "nominal ratio & implied byte frac. & Kendall $\\tau$ & byte-infeasible methods \\\\",
        "\\midrule",
    ]
    for r, bf, tau, infeas in rows:
        tstr = "n/a" if tau is None else f"{tau:.2f}"
        istr = ", ".join(infeas) if infeas else "---"
        lines.append(f"{r:.2f} & {bf:.2f} & {tstr} & {istr} \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    with open(args.out, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"[table1] wrote {args.out}")
    for r, bf, tau, infeas in rows:
        print(f"  ratio {r:.2f}: tau={tau} infeasible={infeas}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/configs/kaggle.yaml
# Kaggle sweep — GPU (T4x2 / P100). Drives BOTH the byte audit (kvbench.run, uses `ratios`) and the
# accuracy re-ranking (kvbench.eval, uses `nominal_ratios`). device_map=auto shards 7B across T4x2.
#
# Qwen models are open (no HF gating). To add Llama-3.1-8B: add it to `models`, attach your HF token
# via Kaggle Secrets, and huggingface-cli login in the notebook.

models:
  - Qwen/Qwen2.5-3B-Instruct
  - Qwen/Qwen2.5-7B-Instruct

device_map: auto          # multi-GPU sharding; omit/none for single-GPU .to(device)

context_lengths:
  - 2048
  - 4096

ratios:                   # byte-audit knob (kvbench.run)
  - 0.25
  - 0.50
  - 0.75
  - 0.90

nominal_ratios:           # accuracy sweep knob (kvbench.eval)
  - 0.10
  - 0.25
  - 0.50
  - 0.75
  - 0.90

methods:
  - random
  - streaming_llm
  - snapkv
  - observed_attention
  - adakv
  - think

n_items: 20


## 3. Sanity: the byte instrument must pass
All three controls, including the masking positive control, must say PASS.

In [ ]:
import os; os.chdir('/kaggle/working')
!PYTHONPATH=/kaggle/working/src python -m kvbench.selftest


## 4. (Optional) edit the config
Uncomment to trim the sweep for a faster first run.

In [ ]:
# import yaml
# cfg = yaml.safe_load(open('/kaggle/working/configs/kaggle.yaml'))
# cfg['models'] = ['Qwen/Qwen2.5-3B-Instruct']   # 3B only
# cfg['context_lengths'] = [2048]
# cfg['n_items'] = 10
# yaml.safe_dump(cfg, open('/kaggle/working/configs/kaggle.yaml','w'))
# print(open('/kaggle/working/configs/kaggle.yaml').read())


## 5. Phase 2 — byte audit (Fig 1). No accuracy eval; the motivating result.

In [ ]:
!PYTHONPATH=/kaggle/working/src python -m kvbench.run --config configs/kaggle.yaml --results results --device cuda


## 6. Phase 3 — accuracy re-ranking at matched bytes (Fig 2, Table 1)

In [ ]:
!PYTHONPATH=/kaggle/working/src python -m kvbench.eval --config configs/kaggle.yaml --results results --device cuda


## 7. Generate figures + table from the real results

In [ ]:
!PYTHONPATH=/kaggle/working/src python scripts/fig1_mispricing.py --results results --out reports/figures/fig1_mispricing.pdf
!PYTHONPATH=/kaggle/working/src python scripts/fig2_reranking.py --results results --out reports/figures/fig2_reranking.pdf
!PYTHONPATH=/kaggle/working/src python scripts/table1_rank.py --results results --out reports/figures/table1_rank.tex
print(open('reports/figures/table1_rank.tex').read())


## 8. Preview Fig 1 inline

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, json, glob
from collections import defaultdict
cells=[json.load(open(p)) for p in glob.glob('results/*.json')]
cells=[c for c in cells if not c.get('failed') and c.get('kind')!='accuracy']
ctx=max(c['context_len'] for c in cells)
cells=[c for c in cells if c['context_len']==ctx]
s=defaultdict(list)
for c in cells: s[c['method']].append((c['nominal_ratio'], c['realized_bytes']/c['full_cache_bytes']))
fig,ax=plt.subplots(figsize=(7,4.3))
ax.plot([0,1],[1,0],'k--',lw=1,label='ratio promises'); ax.axhline(1.0,color='0.6',ls=':')
for m,p in sorted(s.items()):
    p.sort(); ax.plot([x for x,_ in p],[y for _,y in p],marker='o',label=m)
ax.set_xlabel('nominal compression ratio'); ax.set_ylabel('realized bytes / full cache')
ax.set_title(f'Ratios are not bytes (ctx={ctx})'); ax.legend(fontsize=8,loc='center left',bbox_to_anchor=(1.01,0.5))
plt.tight_layout(); plt.show()


## 9. Package results as a downloadable artifact
Everything in `/kaggle/working` persists when you **Save Version**. This also zips the
results + figures so you can download one file (Output tab → `kvbench_artifacts.zip`), or
**Create Dataset** from the output to reuse in another notebook.

In [ ]:
import shutil, glob, os
os.makedirs('/kaggle/working/artifact', exist_ok=True)
shutil.copytree('/kaggle/working/results', '/kaggle/working/artifact/results', dirs_exist_ok=True)
shutil.copytree('/kaggle/working/reports/figures', '/kaggle/working/artifact/figures', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/kvbench_artifacts', 'zip', '/kaggle/working/artifact')
n=len(glob.glob('/kaggle/working/results/*.json'))
print(f'{n} result cells zipped -> /kaggle/working/kvbench_artifacts.zip')
